In [ ]:
#@title Copyright 2019 Google LLC. { display-mode: "form" }
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

<table class="ee-notebook-buttons" align="left"><td>
<a target="_blank"  href="http://colab.research.google.com/github/google/earthengine-community/blob/master/guides/linked/ee-api-colab-setup.ipynb">
    <img src="https://www.tensorflow.org/images/colab_logo_32px.png" /> Run in Google Colab</a>
</td><td>
<a target="_blank"  href="https://github.com/google/earthengine-community/blob/master/guides/linked/ee-api-colab-setup.ipynb"><img width=32px src="https://www.tensorflow.org/images/GitHub-Mark-32px.png" /> View source on GitHub</a></td></table>

# Earth Engine Python API Colab Setup

This notebook demonstrates how to setup the Earth Engine Python API in Colab and provides several examples of how to print and visualize Earth Engine processed data.

## Import API and get credentials

The Earth Engine API is installed by default in Google Colaboratory so requires only importing and authenticating. These steps must be completed for each new Colab session, if you restart your Colab kernel, or if your Colab virtual machine is recycled due to inactivity.

### Import the API

Run the following cell to import the API into your session.

In [1]:
import ee

### Authenticate and initialize

Run the `ee.Authenticate` function to authenticate your access to Earth Engine servers and `ee.Initialize` to initialize it. Upon running the following cell you'll be asked to grant Earth Engine access to your Google account. Follow the instructions printed to the cell.

In [12]:
# Trigger the authentication flow.
ee.Authenticate()

# Initialize the library.
ee.Initialize(project='focal-bucksaw-464620-r8')

In [13]:
from ee import apifunction

def get_roi(lat, lon, buffer_km):
      """
      Generates a bounding box geometry from a center point and buffer size.

      Args:
        lat: Latitude of the center point.
        lon: Longitude of the center point.
        buffer_km: Buffer size in kilometers.

      Returns:
        An ee.Geometry object representing the bounding box.
      """
      point = ee.Geometry.Point(lon, lat)
      buffer = point.buffer(buffer_km * 1000)  # Convert km to meters
      return buffer.bounds()

# Define coordinates for a location (e.g., Boulder, CO)
latitude = 40.0150
longitude = -105.2705
buffer_size_km = 5  # Example buffer size

# Create a bounding box geometry
roi = get_roi(latitude, longitude, buffer_size_km)

In [15]:
# Load a Sentinel-2 image collection
s2 = ee.ImageCollection('COPERNICUS/S2_SR')

In [16]:
s2 = ee.ImageCollection("COPERNICUS/S2_SR") \
    .filterBounds(roi) \
    .filterDate('2023-06-01', '2023-08-31') \
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 10)) \
    .median() \
    .clip(roi)

In [17]:
import geemap

# Create map and center on ROI
Map = geemap.Map(center=[40.0, -105.15], zoom=10)
Map.addLayer(s2, {
    'bands': ['B4', 'B3', 'B2'],
    'min': 0,
    'max': 3000,
    'gamma': 1.2
}, 'Sentinel-2 RGB')
Map

Map(center=[40.0, -105.15], controls=(WidgetControl(options=['position', 'transparent_bg'], widget=SearchDataG…